<a href="https://colab.research.google.com/github/altaseb12/school/blob/master/app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install streamlit pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 68.5 MB/s eta 0:00:00


In [4]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
from itertools import combinations

# ---------------------------------------------------
# PAGE TITLE
# ---------------------------------------------------

st.title("OLAP Data Cube Dashboard")

st.write("This dashboard demonstrates SQL and Python based OLAP operations.")

# ---------------------------------------------------
# SAMPLE DATASET
# ---------------------------------------------------

sales_data = {
    'Product': ['Laptop', 'Laptop', 'Phone', 'Phone', 'Laptop', 'Phone', 'Tablet', 'Tablet'],
    'Region': ['Addis Ababa', 'Bahir Dar', 'Addis Ababa', 'Bahir Dar',
               'Addis Ababa', 'Addis Ababa', 'Adama', 'Bahir Dar'],
    'Year': [2024, 2024, 2024, 2024, 2025, 2025, 2025, 2024],
    'Sales_Amount': [50000, 30000, 20000, 15000, 60000, 25000, 18000, 22000]
}

# Create DataFrame

df = pd.DataFrame(sales_data)

# ---------------------------------------------------
# SQLITE DATABASE CONNECTION
# ---------------------------------------------------

conn = sqlite3.connect('sales.db')

cursor = conn.cursor()

# Save dataframe into SQL table

df.to_sql(
    'Sales',
    conn,
    if_exists='replace',
    index=False
)

# ---------------------------------------------------
# SHOW DATASET
# ---------------------------------------------------

st.header("1. Original Sales Dataset")

st.dataframe(df)

# ---------------------------------------------------
# KPI METRICS
# ---------------------------------------------------

st.header("2. Dashboard KPIs")

col1, col2, col3 = st.columns(3)

col1.metric(
    "Total Sales",
    int(df['Sales_Amount'].sum())
)

col2.metric(
    "Total Products",
    df['Product'].nunique()
)

col3.metric(
    "Total Regions",
    df['Region'].nunique()
)

# ---------------------------------------------------
# SQL QUERY RESULTS
# ---------------------------------------------------

st.header("3. SQL OLAP Query Results")

sql_query = """

SELECT
    Product,
    Region,
    SUM(Sales_Amount) AS Total_Sales

FROM Sales

GROUP BY Product, Region

"""

sql_result = pd.read_sql_query(sql_query, conn)

st.write("SQL GROUP BY Result")

st.dataframe(sql_result)

# ---------------------------------------------------
# SQL ROLL-UP STYLE QUERY
# ---------------------------------------------------

st.header("4. SQL Roll-Up Aggregation")

sql_rollup = """

SELECT
    Product,
    SUM(Sales_Amount) AS Total_Sales

FROM Sales

GROUP BY Product

"""

rollup_result = pd.read_sql_query(sql_rollup, conn)

st.dataframe(rollup_result)

# ---------------------------------------------------
# PYTHON ROLL-UP OPERATION
# ---------------------------------------------------

st.header("5. Python Roll-Up Operation")

rollup = df.groupby('Product')['Sales_Amount'].sum().reset_index()

st.dataframe(rollup)

# ---------------------------------------------------
# DRILL-DOWN OPERATION
# ---------------------------------------------------

st.header("6. Drill-Down Operation")

drilldown = df.groupby(
    ['Product', 'Region', 'Year']
)['Sales_Amount'].sum().reset_index()

st.write("Detailed Sales Analysis")

st.dataframe(drilldown)

# ---------------------------------------------------
# SLICE OPERATION
# ---------------------------------------------------

st.header("7. Slice Operation")

selected_year = st.selectbox(
    "Select Year",
    sorted(df['Year'].unique())
)

slice_data = df[df['Year'] == selected_year]

st.write(f"Sales Data for Year {selected_year}")

st.dataframe(slice_data)

# ---------------------------------------------------
# DICE OPERATION
# ---------------------------------------------------

st.header("8. Dice Operation")

selected_product = st.selectbox(
    "Select Product",
    df['Product'].unique()
)

selected_region = st.selectbox(
    "Select Region",
    df['Region'].unique()
)

dice_data = df[
    (df['Product'] == selected_product) &
    (df['Region'] == selected_region)
]

st.write("Filtered Dice Result")

st.dataframe(dice_data)

# ---------------------------------------------------
# DATA CUBE CONSTRUCTION
# ---------------------------------------------------

st.header("9. OLAP Data Cube Construction")

dimensions = ['Product', 'Region', 'Year']

selected_dimensions = st.multiselect(
    "Select Cube Dimensions",
    dimensions,
    default=['Product', 'Region']
)

if len(selected_dimensions) > 0:

    cube = df.groupby(
        selected_dimensions
    )['Sales_Amount'].sum().reset_index()

    st.write("Generated Data Cube")

    st.dataframe(cube)

else:

    st.warning("Please select at least one dimension")

# ---------------------------------------------------
# GRAND TOTAL
# ---------------------------------------------------

st.header("10. Grand Total Sales")

st.metric(
    label="Total Sales",
    value=int(df['Sales_Amount'].sum())
)

# ---------------------------------------------------
# BAR CHART
# ---------------------------------------------------

st.header("11. Product Wise Sales Bar Chart")

chart_data = df.groupby('Product')['Sales_Amount'].sum()

fig, ax = plt.subplots()

chart_data.plot(kind='bar', ax=ax)

ax.set_title("Product Wise Sales")
ax.set_xlabel("Product")
ax.set_ylabel("Sales Amount")

st.pyplot(fig)

# ---------------------------------------------------
# PIE CHART
# ---------------------------------------------------

st.header("12. Sales Distribution Pie Chart")

fig2, ax2 = plt.subplots()

chart_data.plot(kind='pie', autopct='%1.1f%%', ax=ax2)

ax2.set_ylabel('')

st.pyplot(fig2)

# ---------------------------------------------------
# LINE CHART
# ---------------------------------------------------

st.header("13. Yearly Sales Trend")

yearly_sales = df.groupby('Year')['Sales_Amount'].sum()

st.line_chart(yearly_sales)

# ---------------------------------------------------
# AREA CHART
# ---------------------------------------------------

st.header("14. Area Chart")

st.area_chart(yearly_sales)

# ---------------------------------------------------
# REGION BAR CHART
# ---------------------------------------------------

st.header("15. Region Wise Sales")

region_sales = df.groupby('Region')['Sales_Amount'].sum()

st.bar_chart(region_sales)

# ---------------------------------------------------
# SCATTER PLOT
# ---------------------------------------------------

st.header("16. Scatter Plot")

fig3, ax3 = plt.subplots()

ax3.scatter(
    df['Year'],
    df['Sales_Amount']
)

ax3.set_xlabel("Year")
ax3.set_ylabel("Sales Amount")
ax3.set_title("Sales Scatter Plot")

st.pyplot(fig3)

# ---------------------------------------------------
# HISTOGRAM
# ---------------------------------------------------

st.header("17. Sales Distribution Histogram")

fig4, ax4 = plt.subplots()

ax4.hist(df['Sales_Amount'])

ax4.set_title("Histogram")

st.pyplot(fig4)

# ---------------------------------------------------
# COMPLETE CUBE GENERATION
# ---------------------------------------------------

st.header("18. Complete OLAP Cube Aggregations")

for i in range(len(dimensions) + 1):

    for combo in combinations(dimensions, i):

        st.subheader(f"Cube: {combo}")

        if len(combo) == 0:

            total = df['Sales_Amount'].sum()

            st.write("Grand Total")
            st.write(total)

        else:

            cube_data = df.groupby(
                list(combo)
            )['Sales_Amount'].sum().reset_index()

            st.dataframe(cube_data)

# ---------------------------------------------------
# DOWNLOAD BUTTON
# ---------------------------------------------------

st.header("19. Download SQL Result")

csv = sql_result.to_csv(index=False)

st.download_button(
    label="Download SQL Result CSV",
    data=csv,
    file_name='sql_result.csv',
    mime='text/csv'
)

# ---------------------------------------------------
# FOOTER
# ---------------------------------------------------

st.success("OLAP Dashboard Executed Successfully")

# Close connection

conn.close()

2026-05-26 08:22:10.725 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-26 08:22:10.729 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-26 08:22:10.733 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-26 08:22:10.737 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-26 08:22:10.741 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-26 08:22:10.746 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-26 08:22:10.797 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-26 08:22:10.802 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar